<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/Credit_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ScamGuard-MY: Financial Document Analyzer (PDPA Compliant)

This notebook extracts unstructured text from financial documents — **credit reports, invoices, overdue statements, and purchase orders** — in PDF or spreadsheet form, redacts Personally Identifiable Information (PII) like NRICs, emails, and phone numbers for **PDPA compliance**, auto-detects the document type, pulls out structured fields relevant to that type, flags risks, and presents an AI-powered dashboard.

**Supports:** PDF and spreadsheet (`.xlsx` / `.csv`) input. You can generate mock samples to test with, or upload your own files.

---

## Pipeline Logic Overview

```
┌─────────────┐    ┌───────────────┐    ┌──────────────────┐    ┌───────────────────┐    ┌──────────────┐    ┌───────────────┐
│  INPUT FILE │───▶│  EXTRACT TEXT  │───▶│  MASK PII (PDPA) │───▶│  DETECT DOC TYPE  │───▶│  EXTRACT     │───▶│  DETECT RISKS │
│ PDF/XLSX/CSV│    │ pdfplumber /   │    │  NRIC, Email,    │    │  invoice / PO /   │    │  STRUCTURED  │    │  keyword +    │
│             │    │ pandas         │    │  Phone → REDACTED│    │  credit / overdue │    │  FIELDS      │    │  pattern scan │
└─────────────┘    └───────────────┘    └──────────────────┘    └───────────────────┘    └──────────────┘    └───────────────┘
                                                                                                                       │
                                                                                                                       ▼
                                                                                                              ┌───────────────┐
                                                                                                              │  AI ANALYSIS  │
                                                                                                              │  (LLM / Mock) │
                                                                                                              └───────┬───────┘
                                                                                                                       │
                                                                                                                       ▼
                                                                                                              ┌───────────────┐
                                                                                                              │  DASHBOARD    │
                                                                                                              │  (HTML render)│
                                                                                                              └───────────────┘
```

### Key Design Decisions
1. **Privacy-first:** PII is redacted *before* any analysis or LLM call — the AI never sees raw NRICs/emails/phones.
2. **Format-agnostic:** One pipeline handles PDFs (via `pdfplumber`) and spreadsheets (via `pandas`) through a router function.
3. **Type-aware extraction:** Document type is detected first, then type-specific regex rules pull structured fields.
4. **Defence-in-depth risk detection:** Keyword scanning catches common red flags; the LLM layer provides deeper semantic analysis.

### 1. Install Dependencies

In [ ]:
!pip install pdfplumber openpyxl reportlab -q

### 2. Core Engine — Complete Pipeline (All Functions)

This single cell contains **everything** the pipeline needs:
- PII masking (PDPA compliance)
- Text extraction (PDF + spreadsheet)
- Document type detection
- Structured field extraction (per document type)
- Risk/exception detection
- AI analysis simulation + dashboard renderer

**Run this cell once** — all subsequent cells depend on it. Because everything is defined here, cell execution order no longer matters for the remaining cells.

#### Logic Breakdown:

| Component | What it does | Why |
|-----------|-------------|-----|
| `mask_experian_pii()` | Regex-replaces NRIC, email, phone with `[REDACTED_*]` | PDPA compliance — PII never reaches the LLM |
| `extract_from_spreadsheet()` | Reads `.xlsx`/`.csv` with `header=None`, filters NaN, joins non-empty cells | Handles messy real-world spreadsheets with merged cells |
| `extract_any()` | Routes file to PDF or spreadsheet extractor, then masks PII | Single entry point regardless of file format |
| `detect_document_type()` | Keyword-based classifier (overdue → credit, PO keywords → PO, invoice keywords → invoice) | Determines which field-extraction rules to apply |
| `extract_structured_fields()` | Dispatches to type-specific regex extractors | Pulls key fields (invoice no, dates, totals, etc.) |
| `detect_risks()` | Scans each line for risk keywords | Flags lines needing human attention |
| `analyze_document()` | Orchestrates the full pipeline and returns a result dict | Main entry point for the entire system |
| `simulate_llm_analysis()` | Mock LLM response (replace with real API for production) | Generates risk score + recommendations |
| `render_financial_dashboard()` | Renders HTML dashboard from AI insights | Visual output for end users |

In [ ]:
import re
import pdfplumber
import pandas as pd
from IPython.display import display, HTML


# ═══════════════════════════════════════════════════════════════════════════════
# 2.1  PII Masking (PDPA Compliance Layer)
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: We use three regex patterns to catch Malaysian PII formats:
#   - NRIC: 6 digits (DOB) + dash + 2 digits (state) + dash + 4 digits
#   - Email: standard RFC-like pattern
#   - Phone: Malaysian mobile starting with 01X
# Each match is replaced with a placeholder token so downstream analysis
# never sees the raw PII — this is the PDPA compliance guarantee.

def mask_experian_pii(text: str) -> str:
    """Masks Malaysian NRICs, Emails, and Phone Numbers for PDPA Compliance.
    
    This runs BEFORE any analysis, ensuring PII is never exposed to
    the LLM or stored in analysis results.
    """
    # Malaysian NRIC: YYMMDD-SS-NNNN (e.g. 880412-14-5531)
    nric_pattern = r"\b\d{6}-\d{2}-\d{4}\b"
    # Standard email addresses
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
    # Malaysian mobile: 01X-XXXX-XXXX (with optional dashes/spaces)
    phone_pattern = r"\b01[0-9][-\s]?\d{3,4}[-\s]?\d{4}\b"

    masked = re.sub(nric_pattern, "[REDACTED_NRIC]", text)
    masked = re.sub(email_pattern, "[REDACTED_EMAIL]", masked)
    masked = re.sub(phone_pattern, "[REDACTED_PHONE]", masked)
    return masked


# ═══════════════════════════════════════════════════════════════════════════════
# 2.2  Text Extraction (PDF or Spreadsheet → raw text)
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: Real-world financial documents come in two forms:
#   1. PDFs — use pdfplumber to extract text page by page
#   2. Spreadsheets — trickier because they often have merged cells,
#      free-form text blocks, and no consistent header row.
#
# FIX APPLIED: We now use `header=None` when reading spreadsheets so the
# first row isn't swallowed as column names. We also filter out NaN/empty
# cells before joining, which handles merged-cell layouts gracefully.

def extract_from_spreadsheet(file_path: str) -> str:
    """Extracts and flattens spreadsheet data into text for the pipeline.
    
    Uses header=None to avoid losing the first row as column names.
    Filters NaN values to handle merged cells and sparse layouts.
    """
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path, header=None)
    else:
        # Read ALL sheets if multiple exist, but default to first
        df = pd.read_excel(file_path, header=None)

    lines = []
    for _, row in df.iterrows():
        # Filter out NaN/empty values — critical for merged-cell spreadsheets
        values = [str(v).strip() for v in row if pd.notna(v) and str(v).strip()]
        if values:
            lines.append(" | ".join(values))
    return "\n".join(lines)


def extract_any(file_path: str) -> str:
    """Router: sends the file to the right extractor based on extension, then masks PII.
    
    This is the single entry point for text extraction. The masking happens
    here (not in the caller) so it's impossible to forget the PDPA step.
    """
    if file_path.lower().endswith((".xlsx", ".xls", ".csv")):
        raw_text = extract_from_spreadsheet(file_path)
    else:
        # Assume PDF for anything else
        raw_text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    raw_text += text + "\n"
    return mask_experian_pii(raw_text)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.3  Document Type Detection
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: We classify documents by scanning for distinctive keywords/phrases.
# The order matters — we check from MOST specific to LEAST specific:
#   1. Overdue/collection statements (often contain 'invoice' too, so check first)
#   2. Purchase orders
#   3. Standard invoices
#   4. Credit reports
#   5. Unknown (fallback)
#
# FIX APPLIED: Added detection for overdue/billing statements which were
# previously mis-classified as regular invoices. An overdue statement with
# debt/liability language is now correctly routed to credit_report extraction.

def detect_document_type(sanitized_text: str) -> str:
    """Classifies document type based on keyword signals.
    
    Checked in order of specificity to avoid mis-classification.
    For example, an 'Overdue Invoice' statement contains the word 'invoice'
    but should be treated as a credit/collections document, not a standard invoice.
    """
    text_lower = sanitized_text.lower()

    # --- Priority 1: Overdue / Collection / Billing Statements ---
    # These often mention 'invoice' but are really debt-collection documents.
    # We detect them by the co-occurrence of overdue/collection language WITH
    # debt/liability/lawsuit language.
    overdue_signals = ["overdue", "billing statement", "delinquency", "collection warning",
                       "past due", "outstanding balance", "default"]
    debt_signals = ["liabilities", "debt", "lawsuit", "legal action", "credit default"]
    
    has_overdue = any(signal in text_lower for signal in overdue_signals)
    has_debt = any(signal in text_lower for signal in debt_signals)
    
    if has_overdue and has_debt:
        return "overdue_statement"

    # --- Priority 2: Purchase Orders ---
    if ("purchase order" in text_lower or 
        re.search(r"\bpo\s*number\b", text_lower) or 
        re.search(r"\bpo[\s#\-]*\d", text_lower)):
        return "purchase_order"

    # --- Priority 3: Standard Tax Invoices ---
    if ("tax invoice" in text_lower or 
        "invoice no" in text_lower or 
        "invoice number" in text_lower or 
        re.search(r"\binvoice\b", text_lower)):
        return "invoice"

    # --- Priority 4: Credit Reports / Assessments ---
    if ("credit assessment" in text_lower or 
        "credit report" in text_lower or 
        "experian" in text_lower):
        return "credit_report"

    return "unknown"


# ═══════════════════════════════════════════════════════════════════════════════
# 2.4  Structured Field Extraction (per document type)
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: Once we know the document type, we apply type-specific regex patterns
# to pull out the key fields a finance team cares about.
#
# Each extractor returns a dict of field_name → value (or None if not found).
# The regexes are intentionally loose (using \s* for flexible whitespace) to
# handle variations in how PDFs and spreadsheets render spacing.
#
# FIX APPLIED: Broadened regexes to handle:
#   - Dates in YYYY-MM-DD format (not just DD/MM/YYYY)
#   - Fields separated by pipes (from spreadsheet extraction)
#   - 'TOTAL DUE' without a colon
#   - Added overdue_statement field extractor

def _first_match(pattern, text, group=1, flags=re.IGNORECASE):
    """Helper: returns the first regex match group, or None."""
    m = re.search(pattern, text, flags)
    return m.group(group).strip() if m else None


def extract_invoice_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a standard tax invoice."""
    return {
        "invoice_no": _first_match(
            r"invoice\s*(?:no|number)\.?[:\s]+([A-Za-z0-9\-]+)", sanitized_text),
        "invoice_date": _first_match(
            r"invoice\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "due_date": _first_match(
            r"due\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "bill_to": _first_match(
            r"bill\s*to[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_due": _first_match(
            r"total\s*(?:due|payable)[:\s]*(RM[\s\d,\.]+)", sanitized_text),
        "payment_status": _first_match(
            r"payment\s*status[:\s]+([A-Za-z ]+?)(?:\n|\||$)", sanitized_text),
    }


def extract_po_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a purchase order."""
    return {
        "po_number": _first_match(
            r"po\s*number[:\s]+([A-Za-z0-9\-]+)", sanitized_text),
        "po_date": _first_match(
            r"po\s*date[:\s]+([\d/\-]+)", sanitized_text),
        "expected_delivery": _first_match(
            r"expected\s*delivery[:\s]+([\d/\-]+)", sanitized_text),
        "vendor": _first_match(
            r"vendor[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_po_value": _first_match(
            r"total\s*po\s*value[:\s]*(RM[\s\d,\.]+)", sanitized_text),
        "payment_terms": _first_match(
            r"payment\s*terms[:\s]+([A-Za-z0-9 ]+?)(?:\n|\||$)", sanitized_text),
    }


def extract_credit_report_fields(sanitized_text: str) -> dict:
    """Extracts key fields from a credit assessment report."""
    return {
        "company_name": _first_match(
            r"company\s*name[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "director_name": _first_match(
            r"director\s*(?:name|in[- ]charge)[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_liabilities": _first_match(
            r"(?:total\s*(?:outstanding\s*)?)?(?:liabilities|unpaid)[^\d]*(RM[\s\d,\.]+)",
            sanitized_text),
    }


def extract_overdue_statement_fields(sanitized_text: str) -> dict:
    """Extracts key fields from an overdue billing / collection statement.
    
    These documents are hybrid — they contain invoice-like line items but
    are really debt-collection notices. We extract both the debt details
    and the debtor/creditor information.
    """
    return {
        "creditor": _first_match(
            r"(?:vendor\s*/\s*)?creditor[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "debtor": _first_match(
            r"(?:debtor\s*(?:company)?)[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "total_unpaid": _first_match(
            r"(?:total\s*(?:unpaid|outstanding)\s*(?:liabilities|amount)?)[^\d]*(RM[\s\d,\.]+)",
            sanitized_text),
        "risk_status": _first_match(
            r"(?:account\s*)?risk\s*status[:\s]+(.+?)(?:\n|\|)", sanitized_text),
        "payment_warning": _first_match(
            r"payment\s*warning[:\s]+(.+?)(?:\n|$)", sanitized_text),
    }


def extract_structured_fields(sanitized_text: str, doc_type: str) -> dict:
    """Dispatches to the right field extractor based on detected document type."""
    if doc_type == "invoice":
        return extract_invoice_fields(sanitized_text)
    if doc_type == "purchase_order":
        return extract_po_fields(sanitized_text)
    if doc_type == "credit_report":
        return extract_credit_report_fields(sanitized_text)
    if doc_type == "overdue_statement":
        return extract_overdue_statement_fields(sanitized_text)
    return {}


# ═══════════════════════════════════════════════════════════════════════════════
# 2.5  Risk / Exception Detection
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: We scan every line for known risk-related keywords.
# This is a simple first-pass filter — it won't catch every subtle phrasing,
# but it's fast and deterministic. The LLM layer (Step 6) provides deeper
# semantic risk analysis on top of this.
#
# FIX APPLIED: Expanded the keyword list to catch more real-world phrasing
# like "delinquent", "collections", "arrears", "legal action", etc.

DEFAULT_RISK_KEYWORDS = [
    "lawsuit", "liabilities", "deteriorating", "unpaid", "debt", "risk",
    "overdue", "suspension", "default", "penalty", "dispute", "terminate",
    # --- NEW: additional real-world risk language ---
    "delinquent", "delinquency", "arrears", "collections", "legal action",
    "credit default", "write-off", "write off", "bad debt", "non-performing",
    "past due", "outstanding balance", "service suspension", "warning",
    "high risk", "severe", "forfeiture", "indemnity",
]


def detect_risks(sanitized_text: str, risk_keywords=None) -> list:
    """Flags lines containing risk-related keywords.
    
    Returns a deduplicated list of lines that matched at least one keyword.
    NOTE: This is keyword-based, not semantic — phrasing outside this list
    will not be caught. The LLM analysis layer compensates for this.
    """
    if risk_keywords is None:
        risk_keywords = DEFAULT_RISK_KEYWORDS

    detected_risks = []
    for line in sanitized_text.split('\n'):
        if any(keyword in line.lower() for keyword in risk_keywords):
            if line.strip() and line.strip() not in detected_risks:
                detected_risks.append(line.strip())
    return detected_risks


# ═══════════════════════════════════════════════════════════════════════════════
# 2.6  Full Pipeline Entry Point
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: This orchestrates the entire flow:
#   1. extract_any() → gets text and masks PII
#   2. detect_document_type() → classifies what kind of document this is
#   3. extract_structured_fields() → pulls type-specific key fields
#   4. detect_risks() → flags risk/exception lines
#   5. Returns everything in a dict for the dashboard to consume

def analyze_document(file_path: str) -> dict:
    """Runs the full pipeline: extract → mask PII → detect type → extract fields → detect risks.
    
    Returns a result dict that feeds into the AI analysis and dashboard steps.
    """
    try:
        sanitized_text = extract_any(file_path)
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{file_path}'")
        print("Please make sure you have generated or uploaded the file first (Step 3 or 4).")
        return {}
    except Exception as e:
        print(f"❌ Error reading '{file_path}': {e}")
        return {}

    doc_type = detect_document_type(sanitized_text)
    fields = extract_structured_fields(sanitized_text, doc_type)
    risks = detect_risks(sanitized_text)

    result = {
        "file_path": file_path,
        "document_type": doc_type,
        "sanitized_text": sanitized_text,
        "fields": fields,
        "risks": risks,
    }

    # --- Pretty-print output ---
    print(f"--- FILE: {file_path} ---")
    print(f"Detected Document Type: {doc_type.upper()}\n")

    print("--- PDPA SANITIZED OUTPUT ---")
    print(sanitized_text)

    print("\n--- STRUCTURED FIELDS ---")
    if fields:
        for k, v in fields.items():
            print(f"{k}: {v if v else '(not found)'}")
    else:
        print("(No structured field rules defined for this document type.)")

    print("\n--- DETECTED RISK / EXCEPTION LINES ---")
    if risks:
        for idx, r in enumerate(risks, 1):
            print(f"[{idx}] {r}")
    else:
        print("(No risk keywords matched.)")

    return result


# Backward compatibility alias
def analyze_credit_report(file_path: str):
    return analyze_document(file_path)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.7  AI Analysis Simulation + Dashboard Renderer
# ═══════════════════════════════════════════════════════════════════════════════
# LOGIC: The simulate_llm_analysis() function is a MOCK — it returns a fixed
# response regardless of input. For production, replace with a real API call
# (see Step 7 at the bottom of this notebook).
#
# The dashboard renderer takes the AI output and creates an HTML visualization
# that shows the risk score, summary, extracted fields, and recommendations.
#
# FIX APPLIED: These functions are now defined in the same cell as everything
# else, so the NameError (from running cells out of order) can never happen.

def simulate_llm_analysis(sanitized_text: str, doc_type: str = "credit_report", fields: dict = None) -> dict:
    """
    Simulates sending sanitized text to an LLM for structured risk assessment.

    NOTE: This is a MOCK response for demo purposes. Replace with a real API call
    (see Step 7) for production use where the response should vary based on input.
    """
    prompt = f"""
    Context: You are a corporate financial risk analyst.
    Document type detected: {doc_type}
    Structured fields extracted: {fields}
    Task: Review the following sanitized {doc_type.replace('_', ' ')} and extract key insights.
    Format your response as a JSON object containing a Risk Score (1-100), a Summary,
    and a list of Actionable Recommendations.
    Data: {sanitized_text}
    """

    # --- MOCK RESPONSE (replace with real LLM call for production) ---
    mock_llm_response = {
        "risk_score": 35,
        "risk_category": "Medium to High Risk",
        "executive_summary": (
            "The document shows signs of financial distress or elevated commercial risk "
            "that warrants follow-up before further action is taken."
        ),
        "actionable_recommendations": [
            "Request clarification on the flagged risk item before proceeding.",
            "Implement stricter payment or approval terms for this counterparty.",
            "Require additional documentation or guarantees before commitment."
        ],
        "explainability_flag": (
            "Risk score weighted by the specific risk/exception lines detected in the document."
        )
    }

    return mock_llm_response


def render_financial_dashboard(ai_insights: dict, doc_type: str = "", fields: dict = None):
    """Renders the AI insights as an HTML dashboard in the notebook."""
    score = ai_insights["risk_score"]
    # Color logic: red for high risk (score >= 50), green for lower risk
    score_color = "red" if score >= 50 else "#e6960a" if score >= 30 else "green"
    fields = fields or {}

    fields_html = "".join(
        f"<li><b>{k.replace('_', ' ').title()}:</b> {v if v else '<i>not found</i>'}</li>"
        for k, v in fields.items()
    ) or "<li><i>No structured fields available for this document type.</i></li>"

    dashboard_html = f"""
    <div style="font-family: sans-serif; border: 2px solid #e5e7eb; border-radius: 10px; padding: 20px; max-width: 800px;">
        <h2 style="color: #1f2937; border-bottom: 2px solid #e5e7eb; padding-bottom: 10px;">📊 AI Financial Document Dashboard</h2>
        <p style="color: #6b7280; margin-top: -5px;">Detected Document Type: <b>{doc_type.replace('_', ' ').title() if doc_type else 'Unknown'}</b></p>

        <div style="display: flex; justify-content: space-between; margin-top: 20px;">
            <div style="background-color: #f9fafb; padding: 15px; border-radius: 8px; width: 45%;">
                <h3 style="margin: 0; color: #4b5563;">Risk Score</h3>
                <h1 style="margin: 10px 0; font-size: 48px; color: {score_color};">{score}/100</h1>
                <p style="margin: 0; font-weight: bold; color: {score_color};">{ai_insights['risk_category']}</p>
            </div>

            <div style="width: 50%;">
                <h3 style="margin-top: 0; color: #4b5563;">Executive Summary</h3>
                <p style="color: #374151; line-height: 1.5;">{ai_insights['executive_summary']}</p>
                <p style="font-size: 12px; color: #6b7280;"><i>Model Rationale: {ai_insights['explainability_flag']}</i></p>
            </div>
        </div>

        <h3 style="color: #4b5563; margin-top: 25px;">📄 Extracted Fields</h3>
        <ul style="color: #374151; line-height: 1.6;">
            {fields_html}
        </ul>

        <h3 style="color: #4b5563; margin-top: 25px;">⚡ Actionable Recommendations</h3>
        <ul style="color: #374151; line-height: 1.6;">
            {"".join(f"<li>{item}</li>" for item in ai_insights['actionable_recommendations'])}
        </ul>

        <p style="font-size: 11px; color: #9ca3af; margin-top: 20px; border-top: 1px solid #f3f4f6; padding-top: 10px;">
            🔒 PDPA Notice: This document was processed in-memory only. Detected NRICs, emails, and phone numbers were redacted before analysis and are not stored or logged anywhere.
        </p>
    </div>
    """
    display(HTML(dashboard_html))


print("✅ Core Engine loaded successfully. All functions are ready.")
print("   Proceed to Step 3 (generate mock files) or Step 4 (upload your own).")

### 3. (Option A) Generate Mock Sample Documents
Run this cell to create three simulated PDFs to test the full pipeline against: a **credit report**, an **invoice**, and a **purchase order** — each with artificial Malaysian PII and a risk/exception signal built in.

**Skip this if you'd rather upload your own files in Step 4.**

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def create_mock_credit_report(filename="Credit_Assessment_Pinnacle.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "EXPERIAN COMMERCIAL CREDIT ASSESSMENT")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 700, "Company Name: Pinnacle Tech Solutions")
    c.drawString(100, 680, "Director Name: Ahmad Razak")
    c.drawString(100, 660, "Director NRIC: 880412-14-5531")
    c.drawString(100, 640, "Contact Email: ahmad.razak@pinnacle.com.my")
    c.drawString(100, 620, "Mobile Phone: 012-3456789")
    c.drawString(100, 580, "FINANCIAL SUMMARY & RISK METRICS:")
    c.drawString(100, 560, "- The entity shows flags of deteriorating capital reserves.")
    c.drawString(100, 540, "- High risk exposure detected due to severe unpaid supplier invoices.")
    c.drawString(100, 520, "- Active lawsuit filed by major vendor on 15/04/2026.")
    c.drawString(100, 500, "- Total outstanding liabilities exceed RM 450,000.")
    c.save()
    print(f"✅ Created {filename}")


def create_mock_invoice(filename="Sample_Invoice_INV-2026-0451.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "TAX INVOICE")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "Invoice No: INV-2026-0451")
    c.drawString(100, 690, "Invoice Date: 12/08/2026")
    c.drawString(100, 670, "Due Date: 26/08/2026")
    c.drawString(100, 630, "Bill To: Meridian Trading Sdn Bhd")
    c.drawString(100, 610, "Attn: Siti Nurhaliza binti Ismail")
    c.drawString(100, 590, "Contact Email: siti.ismail@meridiantrading.com.my")
    c.drawString(100, 570, "Contact Phone: 019-8827364")
    c.drawString(100, 550, "Company Reg NRIC (Director): 850627-08-5142")
    c.drawString(100, 510, "ITEMS:")
    c.drawString(100, 495, "1. Industrial Packaging Rolls x 500 units - RM 8,500.00")
    c.drawString(100, 480, "2. Freight & Logistics Fee            - RM 620.00")
    c.drawString(100, 465, "3. Handling Surcharge                 - RM 150.00")
    c.drawString(100, 430, "Subtotal: RM 9,270.00")
    c.drawString(100, 415, "SST (6%): RM 556.20")
    c.drawString(100, 400, "TOTAL DUE: RM 9,826.20")
    c.drawString(100, 365, "Payment Status: UNPAID")
    c.drawString(100, 350, "Note: This is the third overdue reminder. Outstanding balance")
    c.drawString(100, 335, "has exceeded 45 days past due date. Risk of service suspension")
    c.drawString(100, 320, "if payment is not received within 7 days.")
    c.save()
    print(f"✅ Created {filename}")


def create_mock_po(filename="Sample_PO_PO-8842.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "PURCHASE ORDER")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "PO Number: PO-8842")
    c.drawString(100, 690, "PO Date: 05/08/2026")
    c.drawString(100, 670, "Expected Delivery: 20/08/2026")
    c.drawString(100, 630, "Vendor: Apex Steelworks Sdn Bhd")
    c.drawString(100, 610, "Vendor Contact: Ahmad Faiz bin Zulkifli")
    c.drawString(100, 590, "Vendor Email: faiz.zulkifli@apexsteel.com.my")
    c.drawString(100, 570, "Vendor Phone: 012-5563981")
    c.drawString(100, 550, "Vendor Director NRIC: 780315-10-6231")
    c.drawString(100, 510, "ORDERED ITEMS:")
    c.drawString(100, 495, "1. Galvanized Steel Sheets 2mm x 200 units - RM 24,000.00")
    c.drawString(100, 480, "2. Mounting Brackets x 400 units           - RM 3,200.00")
    c.drawString(100, 465, "3. Delivery & Installation Service         - RM 1,500.00")
    c.drawString(100, 430, "TOTAL PO VALUE: RM 28,700.00")
    c.drawString(100, 395, "Payment Terms: Net 30")
    c.drawString(100, 380, "Vendor Risk Note: Vendor has an active lawsuit filed by a")
    c.drawString(100, 365, "former subcontractor regarding unpaid labour claims. Approve")
    c.drawString(100, 350, "with caution and require signed indemnity before deposit release.")
    c.save()
    print(f"✅ Created {filename}")


create_mock_credit_report()
create_mock_invoice()
create_mock_po()

# Pick which one to run through the pipeline below (change this to test a different file)
sample_file_path = "Sample_Invoice_INV-2026-0451.pdf"

### 4. (Option B) Upload Your Own Document
Run this cell instead of Step 3 to test with a real PDF or spreadsheet (`.xlsx` / `.csv`) — a credit report, invoice, or purchase order. Opens a native Colab file picker.

**If you already ran Step 3, running this too will overwrite `sample_file_path` with your uploaded file.**

In [ ]:
from google.colab import files

print("📤 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...")
uploaded = files.upload()

if uploaded:
    sample_file_path = list(uploaded.keys())[0]
    print(f"✅ '{sample_file_path}' uploaded successfully.")
else:
    print("⚠️ No file uploaded — keeping the existing sample_file_path (if set).")

### 5. Run the Extraction + Detection Pipeline
This runs `analyze_document()` on `sample_file_path`: extracts text, redacts PII, detects the document type, pulls structured fields for that type, and flags risk/exception lines.

Change `sample_file_path` above (Step 3 or 4) to test a different document, then re-run this cell.

In [ ]:
result = analyze_document(sample_file_path)

### 6. Run the Full Pipeline (Extraction + AI Dashboard)

This re-runs extraction on `sample_file_path`, then feeds the sanitized text, detected document type, and structured fields into the AI analysis step and renders the full dashboard.

> ⚠️ **Note:** `simulate_llm_analysis()` currently returns a **hardcoded mock response** — it does not actually read the sanitized text or the structured fields from Step 5. This is fine for testing the pipeline/UI, but for a real submission you should replace it with a genuine LLM API call (see Step 7) so the risk score and summary actually reflect the uploaded document.

In [ ]:
# Re-run extraction (in case sample_file_path changed)
result = analyze_document(sample_file_path)

if result:
    print("\n🤖 Sending sanitized data to AI Engine...\n")
    ai_results = simulate_llm_analysis(
        result["sanitized_text"],
        doc_type=result["document_type"],
        fields=result["fields"],
    )
    render_financial_dashboard(
        ai_results,
        doc_type=result["document_type"],
        fields=result["fields"]
    )
else:
    print("⚠️ No result to analyze — check that sample_file_path is set correctly.")

### 7. (For Your Final Submission) Replace the Mock LLM Call

Right now `simulate_llm_analysis()` returns a **fixed dictionary** no matter what document, type, or fields are passed in. If judges upload a different invoice, PO, or credit report and see the same score and summary every time, that will hurt you on the "accuracy and quality of insights" success criterion.

To fix this, replace the body of `simulate_llm_analysis()` (in Step 2 above) with a real API call that actually uses `doc_type` and `fields`, e.g.:

```python
import anthropic
import json

client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def simulate_llm_analysis(sanitized_text: str, doc_type: str = "credit_report", fields: dict = None) -> dict:
    prompt = f'''
    Context: You are a corporate financial risk analyst.
    Document type: {doc_type}
    Structured fields already extracted: {fields}
    Task: Review the following sanitized {doc_type.replace('_', ' ')} and extract key insights,
    grounding your assessment in the structured fields and any risk/exception language present.
    Respond ONLY with a JSON object (no markdown, no preamble) with keys:
    risk_score (1-100 integer), risk_category (string), executive_summary (string),
    actionable_recommendations (list of strings), explainability_flag (string).
    Data: {sanitized_text}
    '''

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    raw_text = response.content[0].text.strip()
    raw_text = raw_text.replace("```json", "").replace("```", "").strip()
    return json.loads(raw_text)
```

You'd need to `!pip install anthropic` and store your API key securely (e.g. via Colab's Secrets manager, `google.colab.userdata`) rather than hardcoding it. Swap in Gemini's SDK instead if that's the model your team is using — the surrounding pipeline (extraction, masking, doc-type detection, field extraction, dashboard rendering) stays exactly the same either way.

---

**Also worth strengthening:** `detect_risks()` in Step 2 is a keyword match — it will only catch phrasing that literally contains the listed words. A real invoice using different wording will slip through. If you have time, consider having the real LLM call do the risk/exception detection directly from `sanitized_text`, instead of relying on the keyword list — that alone would meaningfully improve accuracy.